# TFT — prévision de charge RTE sur Colab / Kaggle

Backtest walk-forward du **Temporal Fusion Transformer** (implémentation compacte PyTorch du projet), mêmes folds et
mêmes scénarios météo que le benchmark XGBoost/RF/SARIMAX. Sortie : `predictions_tft.parquet` à rapatrier puis fusionner
localement avec `scripts/06_merge_tft.py`.

**Avant de commencer** — en local, depuis le dossier du projet, créez le paquet à uploader (quelques Mo) :
```bash
zip -r rte_bundle.zip src configs scripts data/processed/hourly.parquet pyproject.toml README.md \
    -x '*__pycache__*' '*.egg-info*'
```
Runtime Colab : **Exécution → Modifier le type d'exécution → GPU (T4)**.

⚠️ `--issue-hour` doit être **identique** à celui du benchmark que vous comparez (9 = prévision émise à D-1 10:00,
absent = D 00:00, l'ancienne convention optimiste). `06_merge_tft.py` refuse la fusion sinon.

In [ ]:
!nvidia-smi -L
import torch, pandas as pd, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| pandas", pd.__version__)

## 1. Dossier de sortie (Drive recommandé : Colab peut couper la session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/rte_tft/issue10h'      # <- changez si besoin
!mkdir -p "$OUT"

## 2. Code + données du projet
Uploadez `rte_bundle.zip` (ou placez-le sur Drive et adaptez le chemin).

In [ ]:
import os, zipfile
BUNDLE = '/content/drive/MyDrive/rte_tft/rte_bundle.zip'      # si déjà sur Drive
if not os.path.exists(BUNDLE):
    from google.colab import files
    up = files.upload()                                         # sélectionnez rte_bundle.zip
    BUNDLE = next(iter(up))
os.makedirs('/content/rte', exist_ok=True)
zipfile.ZipFile(BUNDLE).extractall('/content/rte')
%cd /content/rte
!ls

## 3. Installation
`--no-deps` : on garde les versions de Colab (torch, pandas, xgboost, statsmodels y sont déjà). Seuls `holidays` et
`pyarrow` manquent parfois. Si une erreur pandas apparaît au test de fumée, le projet a été développé avec pandas 3 :
`!pip install -q -U "pandas>=3"` puis **Exécution → Redémarrer la session**.

In [ ]:
!pip install -q holidays pyarrow pyyaml joblib
!pip install -q -e . --no-deps
import rte_forecast, pandas as pd
print("rte_forecast OK | pandas", pd.__version__)

## 4. Test de fumée (≈ 1-2 min) : 1 fold, 2 epochs — à lancer AVANT le vrai run

In [ ]:
!python scripts/05_train_tft.py --quick --issue-hour 9 --out-dir /content/tft_smoke

## 5. Run complet
- `--weather normal` : scénario opérationnel (T° prévue = normale). Ajoutez `noisy` (et `perfect`, oracle) ensuite.
- `--resume` : après une coupure, relancez la même cellule, les folds finis sont ignorés.
- `--amp` : précision mixte fp16 (plus rapide sur T4). `--seeds 3` : ensemble de 3 graines (×3 en temps).

Ordre de grandeur sur T4 : 1-3 min par fit, donc ~30-40 min pour 12 folds × 1 scénario × 1 graine.

In [ ]:
!python scripts/05_train_tft.py --weather normal --issue-hour 9 --seeds 1 --amp --resume --out-dir "$OUT"

## 6. Coup d'œil aux résultats + téléchargement

In [ ]:
import pandas as pd
p = pd.read_parquet(f"{OUT}/predictions_tft.parquet")
s = pd.read_parquet(f"{OUT}/tft_fold_stats.parquet")
print(s[["weather_mode","fold_id","fold_mae","fit_seconds","best_epoch"]].round(1).to_string(index=False))
t = p[p.split == "test"]
print("\nMAE test 2024 :", (t.groupby("weather_mode").error.apply(lambda e: e.abs().mean())).round(0).to_dict())
cov = ((t.y_true >= t.q10) & (t.y_true <= t.q90)).mean()
print(f"couverture de l'intervalle P10-P90 sur le test : {cov:.1%} (cible ~80 %)")

In [ ]:
!cd "$OUT" && zip -qr /content/tft_results.zip predictions_tft.parquet tft_fold_stats.parquet weights
from google.colab import files
files.download('/content/tft_results.zip')

## 7. En local
```bash
unzip tft_results.zip -d ~/Downloads/tft_results
python scripts/06_merge_tft.py --tft-dir ~/Downloads/tft_results --results-dir data/results_issue10h
RTE_RESULTS_DIR=data/results_issue10h streamlit run app/dashboard.py
```